In [146]:
# cell 1 - imports and basic config
import sys
import os
from pathlib import Path

# find project root reliably regardless of where jupyter was launched from
# we look for the folder that contains both 'src' and 'data' directories
notebook_dir = Path(os.getcwd())
print(f"Current working directory: {notebook_dir}")

# walk up until we find project root
project_root = notebook_dir
for _ in range(3):  # check current folder and up to 2 parents
    if (project_root / "src").exists() and (project_root / "data").exists():
        break
    project_root = project_root.parent

print(f"Project root found: {project_root}")

# add project root to path so 'from src.config import ...' works
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# verify src is now findable
print(f"src/config.py exists: {(project_root / 'src' / 'config.py').exists()}")

Current working directory: d:\OCR Project\notebooks
Project root found: d:\OCR Project
src/config.py exists: True


In [147]:
import fitz
import pytesseract
import cv2
import numpy as np
from PIL import Image
import io

from src.config import (
    TESSERACT_PATH, OCR_DPI, NATIVE_TEXT_MIN_CHARS,
    PDF_DIR, DEV_PAGE_LIMIT
)

pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH

PDF_PATH = PDF_DIR / "latf-specimen-early-1970s.pdf"

print(f"PDF path: {PDF_PATH}")
print(f"PDF exists: {PDF_PATH.exists()}")
print(f"OCR DPI: {OCR_DPI}")
print(f"Dev page limit: {DEV_PAGE_LIMIT}")

PDF path: d:\OCR Project\notebooks\..\data\pdfs\latf-specimen-early-1970s.pdf
PDF exists: True
OCR DPI: 200
Dev page limit: 20


In [148]:
# cell 2 - open the pdf and see what we're working with
# this tells us how many pages exist and whether native text extraction works
doc = fitz.open(PDF_PATH)

print(f"Total pages: {len(doc)}")
print(f"Processing first {DEV_PAGE_LIMIT} pages in dev mode")
print()

# peek at first 3 pages
# native text = text already embedded in PDF by ABBYY scanner
# if char count is low, page is probably a pure image and needs OCR
for page_num in range(min(3, len(doc))):
    page = doc[page_num]
    native_text = page.get_text().strip()
    char_count = len(native_text)
    
    # replace newlines for cleaner preview
    preview = native_text[:150].replace('\n', ' ')
    
    print(f"Page {page_num + 1}: {char_count} chars")
    print(f"  Preview: {preview}")
    print(f"  Decision: {'NATIVE TEXT ok' if char_count >= NATIVE_TEXT_MIN_CHARS else 'FALLBACK TO OCR'}")
    print()

Total pages: 100
Processing first 20 pages in dev mode

Page 1: 0 chars
  Preview: 
  Decision: FALLBACK TO OCR

Page 2: 0 chars
  Preview: 
  Decision: FALLBACK TO OCR

Page 3: 0 chars
  Preview: 
  Decision: FALLBACK TO OCR



In [149]:
# cell 3 - image preprocessing + OCR function
# this is the core of brick 1
# takes a PDF page, converts to image, cleans it up, runs tesseract

def preprocess_page_image(page, dpi=OCR_DPI):
    # step 1: render the PDF page to a pixel image at our chosen DPI
    # zoom factor = DPI / 72 (72 is PDF's default points per inch)
    # at 200 DPI: zoom = 200/72 = 2.78x the default size
    zoom = dpi / 72
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)
    
    # step 2: convert PyMuPDF's pixmap to a PIL Image
    # we need PIL format for pytesseract
    img_bytes = pix.tobytes("png")
    pil_image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    
    # step 3: convert to OpenCV format (numpy array, BGR color order)
    # OpenCV uses BGR, PIL uses RGB - so we flip the channels
    cv_image = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    
    # step 4: convert to grayscale
    # OCR works on grayscale - color information is irrelevant and adds noise
    gray = cv2.cvtColor(cv_image, cv2.COLOR_BGR2GRAY)
    
    # step 5: Otsu binarization - converts gray pixels to pure black or white
    # Otsu automatically finds the best threshold value
    # this is the single most important preprocessing step for OCR accuracy
    # tradeoff: aggressive binarization can lose faint text on aged paper
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # step 6: convert back to PIL for pytesseract
    preprocessed_pil = Image.fromarray(binary)
    
    return pil_image, preprocessed_pil  # return both: original for CLIP, preprocessed for OCR


def run_ocr(preprocessed_pil):
    # run tesseract in two modes simultaneously:
    
    # mode A: get the plain text string
    # PSM 3 = fully automatic page segmentation (tesseract decides layout)
    # good default for mixed pages with text blocks, tables, headings
    ocr_text = pytesseract.image_to_string(
        preprocessed_pil,
        config='--psm 3'
    )
    
    # mode B: get word-level bounding boxes
    # same PSM, but returns a dataframe with x,y,w,h per detected word
    # we need these coordinates for region-level retrieval later
    # output_type=Dict gives us a Python dict directly, no pandas needed
    ocr_data = pytesseract.image_to_data(
        preprocessed_pil,
        config='--psm 3',
        output_type=pytesseract.Output.DICT
    )
    
    # extract only the words with confidence > 0
    # tesseract returns -1 confidence for non-text regions (whitespace, separators)
    # filtering these out keeps our bounding box data clean
    boxes = []
    for i, word in enumerate(ocr_data['text']):
        conf = int(ocr_data['conf'][i])
        if conf > 0 and word.strip():
            boxes.append({
                'word': word,
                'x': ocr_data['left'][i],
                'y': ocr_data['top'][i],
                'w': ocr_data['width'][i],
                'h': ocr_data['height'][i],
                'conf': conf
            })
    
    # compute a quality score = average confidence across all detected words
    # range 0-100, higher is better
    # we use this later to flag low-quality pages in the UI
    if boxes:
        quality_score = sum(b['conf'] for b in boxes) / len(boxes) / 100
    else:
        quality_score = 0.0
    
    return ocr_text, boxes, quality_score


print("Functions defined: preprocess_page_image, run_ocr")
print("Ready to test on page 1")

Functions defined: preprocess_page_image, run_ocr
Ready to test on page 1


In [150]:
# cell 4 - test OCR on page 1
# this is the moment of truth - does our preprocessing + tesseract actually work?

import time

# grab page 1 (index 0)
page = doc[2]

print("Step 1: Preprocessing page image...")
start = time.time()
original_pil, preprocessed_pil = preprocess_page_image(page, dpi=OCR_DPI)
print(f"  Done in {time.time() - start:.2f}s")
print(f"  Original image size: {original_pil.size}")
print(f"  Preprocessed image size: {preprocessed_pil.size}")

print("\nStep 2: Running OCR...")
start = time.time()
ocr_text, boxes, quality_score = run_ocr(preprocessed_pil)
print(f"  Done in {time.time() - start:.2f}s")
print(f"  Characters extracted: {len(ocr_text)}")
print(f"  Words with bounding boxes: {len(boxes)}")
print(f"  Quality score: {quality_score:.2f} (0=garbage, 1=perfect)")

print("\nFirst 300 chars of OCR text:")
print("-" * 40)
print(ocr_text[:300])
print("-" * 40)

print("\nFirst 3 bounding boxes:")
for box in boxes[:3]:
    print(f"  '{box['word']}' at x={box['x']}, y={box['y']}, conf={box['conf']}")

Step 1: Preprocessing page image...
  Done in 0.40s
  Original image size: (915, 1924)
  Preprocessed image size: (915, 1924)

Step 2: Running OCR...
  Done in 2.79s
  Characters extracted: 3515
  Words with bounding boxes: 534
  Quality score: 0.79 (0=garbage, 1=perfect)

First 300 chars of OCR text:
----------------------------------------
LOS ANGELES TYPE FOUNDERS, INC.

INDEX

Display lines thru Page 30 are 18 point, whenever possible.

A

Admiral Script No.8t0.........
Adonis No. 831

Adscript No. 425
Albertus No. 481..
Albertus Titling No. ‘324. see
Alternate Gothic No.1 No. 51 vee B

Alternate Gothic No.2 No. 77 8
Alternate Gothi
----------------------------------------

First 3 bounding boxes:
  'LOS' at x=176, y=163, conf=95
  'ANGELES' at x=250, y=163, conf=95
  'TYPE' at x=404, y=164, conf=96


In [151]:
# cell 5 - diagnose why OCR is returning nothing
# we need to SEE what tesseract is actually looking at

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for jupyter
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# left: original page image
axes[0].imshow(original_pil)
axes[0].set_title("Original (what we see)")
axes[0].axis('off')

# right: preprocessed (what tesseract sees)
axes[1].imshow(preprocessed_pil, cmap='gray')
axes[1].set_title("After Otsu binarization (what tesseract sees)")
axes[1].axis('off')

plt.tight_layout()
plt.savefig('../data/debug_page1.png', dpi=100, bbox_inches='tight')
plt.close()

print("Saved to D:\\OCR Project\\data\\debug_page1.png")
print("Open that file and tell me:")
print("  - Is the text BLACK on WHITE background? (good)")
print("  - Is the text WHITE on BLACK background? (inverted - bad)")
print("  - Is everything one solid color? (threshold failed completely)")

# also check raw pixel values to diagnose without opening the image
import numpy as np
arr = np.array(preprocessed_pil)
print(f"\nPixel value stats:")
print(f"  Min: {arr.min()} Max: {arr.max()}")
print(f"  Mean: {arr.mean():.1f}")
print(f"  % white pixels (255): {(arr == 255).mean() * 100:.1f}%")
print(f"  % black pixels (0): {(arr == 0).mean() * 100:.1f}%")

Saved to D:\OCR Project\data\debug_page1.png
Open that file and tell me:
  - Is the text BLACK on WHITE background? (good)
  - Is the text WHITE on BLACK background? (inverted - bad)
  - Is everything one solid color? (threshold failed completely)

Pixel value stats:
  Min: 0 Max: 255
  Mean: 233.5
  % white pixels (255): 91.6%
  % black pixels (0): 8.4%


In [152]:
pip install matplotlib==3.9.2


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [153]:
import importlib
import src.config
importlib.reload(src.config)
from src.config import MIN_PAGE_CHARS, MIN_QUALITY_SCORE
print(f"MIN_PAGE_CHARS: {MIN_PAGE_CHARS}")
print(f"MIN_QUALITY_SCORE: {MIN_QUALITY_SCORE}")

MIN_PAGE_CHARS: 50
MIN_QUALITY_SCORE: 0.1


In [154]:
# cell 5 - batch process first 20 pages (fixed version)

import time
from src.config import MIN_PAGE_CHARS, MIN_QUALITY_SCORE  # moved outside loop

pages_data = []

start_page = 1
end_page = min(start_page + DEV_PAGE_LIMIT, len(doc))

print(f"Processing pages {start_page + 1} to {end_page} of {len(doc)}")
print(f"MIN_PAGE_CHARS: {MIN_PAGE_CHARS}, MIN_QUALITY_SCORE: {MIN_QUALITY_SCORE}")
print()

total_start = time.time()

for page_idx in range(start_page, end_page):
    page = doc[page_idx]
    page_num = page_idx + 1

    native_text = page.get_text().strip()

    if len(native_text) >= MIN_PAGE_CHARS:
        ocr_text = native_text
        boxes = []
        quality_score = 1.0
        text_source = "native"
        print(f"Page {page_num}: native ({len(native_text)} chars)")
    else:
        original_pil, preprocessed_pil = preprocess_page_image(page, dpi=OCR_DPI)
        ocr_text, boxes, quality_score = run_ocr(preprocessed_pil)
        text_source = "ocr"
        print(f"Page {page_num}: ocr ({len(ocr_text)} chars, quality={quality_score:.2f})")

    # always render page image for CLIP
    zoom = OCR_DPI / 72
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)
    img_bytes = pix.tobytes("png")
    page_image = Image.open(io.BytesIO(img_bytes)).convert("RGB")

    # skip low quality pages
    if len(ocr_text.strip()) < MIN_PAGE_CHARS and quality_score < MIN_QUALITY_SCORE:
        print(f"  SKIPPED - insufficient content")
        continue

    page_dict = {
        "page_num": page_num,
        "text_source": text_source,
        "native_text": native_text,
        "ocr_text": ocr_text,
        "ocr_boxes": boxes,
        "quality_score": quality_score,
        "page_image": page_image,
        "image_size": page_image.size,
    }

    pages_data.append(page_dict)

total_time = time.time() - total_start
print(f"\nDone. Processed {len(pages_data)} pages in {total_time:.1f}s")
print(f"Average: {total_time/max(len(pages_data),1):.1f}s per page")
print(f"\nQuality distribution:")
scores = [p['quality_score'] for p in pages_data]
print(f"  High (>0.7):      {sum(1 for s in scores if s > 0.7)} pages")
print(f"  Medium (0.4-0.7): {sum(1 for s in scores if 0.4 <= s <= 0.7)} pages")
print(f"  Low (<0.4):       {sum(1 for s in scores if s < 0.4)} pages")

Processing pages 2 to 21 of 100
MIN_PAGE_CHARS: 50, MIN_QUALITY_SCORE: 0.1

Page 2: ocr (0 chars, quality=0.00)
  SKIPPED - insufficient content
Page 3: ocr (3515 chars, quality=0.79)
Page 4: ocr (3017 chars, quality=0.82)
Page 5: ocr (2837 chars, quality=0.73)
Page 6: ocr (1195 chars, quality=0.79)
Page 7: ocr (1241 chars, quality=0.78)
Page 8: ocr (1078 chars, quality=0.83)
Page 9: ocr (1744 chars, quality=0.73)
Page 10: ocr (1826 chars, quality=0.74)
Page 11: ocr (1512 chars, quality=0.78)
Page 12: ocr (1407 chars, quality=0.75)
Page 13: ocr (1518 chars, quality=0.77)
Page 14: ocr (1257 chars, quality=0.77)
Page 15: ocr (1797 chars, quality=0.77)
Page 16: ocr (1133 chars, quality=0.79)
Page 17: ocr (1406 chars, quality=0.77)
Page 18: ocr (1105 chars, quality=0.78)
Page 19: ocr (1202 chars, quality=0.80)
Page 20: ocr (1123 chars, quality=0.79)
Page 21: ocr (1740 chars, quality=0.73)

Done. Processed 19 pages in 60.2s
Average: 3.2s per page

Quality distribution:
  High (>0.7):      1

In [155]:
# cell 6 - chunk the OCR text from each page
# we split each page's text into overlapping chunks
# each chunk carries its page number so we can cite sources later
# overlap means a sentence split across a chunk boundary still gets retrieved

def chunk_page_text(page_dict, chunk_size=400, overlap=80):
    text = page_dict["ocr_text"].strip()
    page_num = page_dict["page_num"]
    
    if not text:
        return []
    
    # split into words first - cleaner than character splits
    words = text.split()
    
    if not words:
        return []
    
    chunks = []
    start = 0
    chunk_idx = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words)
        
        # skip chunks that are too short to be meaningful
        if len(chunk_text) > 30:
            chunks.append({
                "chunk_id": f"p{page_num}_c{chunk_idx}",
                "page_num": page_num,
                "text": chunk_text,
                "text_source": page_dict["text_source"],
                "quality_score": page_dict["quality_score"],
                "char_count": len(chunk_text),
                "word_count": len(chunk_words),
            })
            chunk_idx += 1
        
        # move forward by chunk_size minus overlap
        # overlap means consecutive chunks share some words
        # this prevents answers from falling into the gap between chunks
        start += chunk_size - overlap
    
    return chunks


# build all chunks from all pages
all_chunks = []
for page_dict in pages_data:
    page_chunks = chunk_page_text(page_dict, chunk_size=400, overlap=80)
    all_chunks.extend(page_chunks)

print(f"Total chunks: {len(all_chunks)}")
print(f"From {len(pages_data)} pages")
print(f"Average chunks per page: {len(all_chunks)/len(pages_data):.1f}")
print()
print("Sample chunk:")
print("-" * 40)
print(f"ID: {all_chunks[0]['chunk_id']}")
print(f"Page: {all_chunks[0]['page_num']}")
print(f"Words: {all_chunks[0]['word_count']}")
print(f"Text: {all_chunks[0]['text'][:200]}")
print("-" * 40)

Total chunks: 22
From 19 pages
Average chunks per page: 1.2

Sample chunk:
----------------------------------------
ID: p3_c0
Page: 3
Words: 400
Text: LOS ANGELES TYPE FOUNDERS, INC. INDEX Display lines thru Page 30 are 18 point, whenever possible. A Admiral Script No.8t0......... Adonis No. 831 Adscript No. 425 Albertus No. 481.. Albertus Titling N
----------------------------------------


In [156]:
# cell 6 - chunking with LangChain's recursive splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    # try splitting on these separators in order
    # \n\n = paragraphs, \n = lines, " " = words, "" = characters
    separators=["\n\n", "\n", " ", ""],
    chunk_size=200,        # max characters per chunk
    chunk_overlap=40,      # overlap between consecutive chunks
    length_function=len,
)

all_chunks = []

for page_dict in pages_data:
    text = page_dict["ocr_text"].strip()
    if not text:
        continue
    
    # LangChain splitter returns list of strings
    raw_chunks = splitter.split_text(text)
    
    for idx, chunk_text in enumerate(raw_chunks):
        if len(chunk_text) < 30:
            continue
        all_chunks.append({
            "chunk_id": f"p{page_dict['page_num']}_c{idx}",
            "page_num": page_dict["page_num"],
            "text": chunk_text,
            "text_source": page_dict["text_source"],
            "quality_score": page_dict["quality_score"],
            "char_count": len(chunk_text),
        })

print(f"Total chunks: {len(all_chunks)}")
print(f"From {len(pages_data)} pages")
print(f"Average chunks per page: {len(all_chunks)/len(pages_data):.1f}")
print()
print("Sample chunk:")
print("-" * 40)
print(f"ID: {all_chunks[0]['chunk_id']}")
print(f"Page: {all_chunks[0]['page_num']}")
print(f"Text: {all_chunks[0]['text'][:200]}")
print("-" * 40)

Total chunks: 225
From 19 pages
Average chunks per page: 11.8

Sample chunk:
----------------------------------------
ID: p3_c0
Page: 3
Text: LOS ANGELES TYPE FOUNDERS, INC.

INDEX

Display lines thru Page 30 are 18 point, whenever possible.

A

Admiral Script No.8t0.........
Adonis No. 831
----------------------------------------


In [157]:
# cell 7 - embed all chunks and store in Chroma
# bge-small converts each chunk to a 384-dim vector
# chroma stores vectors + metadata so we can retrieve by similarity later

from sentence_transformers import SentenceTransformer
import chromadb
from src.config import CHROMA_DIR, TEXT_EMBED_MODEL

# load embedding model (already cached from test_setup.py)
print("Loading embedding model...")
text_embedder = SentenceTransformer(TEXT_EMBED_MODEL)
print(f"Model loaded: {TEXT_EMBED_MODEL}")

# init Chroma with persistent storage
# persistent = survives notebook restarts, no re-embedding needed
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# delete collection if it exists (clean slate for dev)
try:
    chroma_client.delete_collection("text_chunks")
    print("Deleted existing text_chunks collection")
except:
    pass

# create fresh collection
# cosine distance = standard for text embeddings
text_collection = chroma_client.create_collection(
    name="text_chunks",
    metadata={"hnsw:space": "cosine"}
)
print("Created text_chunks collection")

# embed all chunks in one batch
# batching is faster than one-by-one because the model processes multiple inputs in parallel
print(f"\nEmbedding {len(all_chunks)} chunks...")
start = time.time()

texts = [c["text"] for c in all_chunks]
embeddings = text_embedder.encode(
    texts,
    batch_size=32,        # process 32 chunks at once
    show_progress_bar=True,
    normalize_embeddings=True  # normalize for cosine similarity
)

print(f"Embedding done in {time.time() - start:.1f}s")
print(f"Embedding matrix shape: {embeddings.shape}")  # should be (115, 384)

# store in Chroma
# Chroma needs: ids (unique strings), embeddings, documents (text), metadatas (dicts)
print("\nStoring in Chroma...")
text_collection.add(
    ids=[c["chunk_id"] for c in all_chunks],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[{
        "page_num": c["page_num"],
        "text_source": c["text_source"],
        "quality_score": c["quality_score"],
        "char_count": c["char_count"],
    } for c in all_chunks]
)

print(f"Stored {text_collection.count()} chunks in Chroma")
print("\nTest retrieval:")
test_results = text_collection.query(
    query_embeddings=text_embedder.encode(
        ["what fonts are available"],
        normalize_embeddings=True
    ).tolist(),
    n_results=3
)
for i, doc in enumerate(test_results['documents'][0]):
    page = test_results['metadatas'][0][i]['page_num']
    print(f"  Result {i+1} (page {page}): {doc[:80]}...")

Loading embedding model...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Model loaded: BAAI/bge-small-en-v1.5
Deleted existing text_chunks collection
Created text_chunks collection

Embedding 225 chunks...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Embedding done in 4.7s
Embedding matrix shape: (225, 384)

Storing in Chroma...


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Stored 225 chunks in Chroma

Test retrieval:
  Result 1 (page 10): 30-pt. -ll pt. 3A

6A. a 784.
Alternates Included in fonts, Sizes 14 ta 36 ‘pain...
  Result 2 (page 11): News Gothic Bold Extended No. 210

News Gothic Bold Extended A

Caps L/e Lic
*10...
  Result 3 (page 18): Goudy Bold No. 294

Goudy Bold ABCDEF 1234

Goudy Bold Italic No. 2941

Goudy Bo...


In [158]:
# quality test cell - ask real questions and see what comes back
# this tells us if retrieval is actually working or just returning garbage

test_queries = [
    "what fonts are available",           # broad - should hit index pages
    "Cooper Black",                        # specific font name - should be exact
    "shipping terms minimum order",        # prose section - should hit front matter
    "Century Schoolbook sizes",            # specific font + metadata
    "script typefaces",                    # category search - semantic
]

print("=" * 60)
for query in test_queries:
    print(f"\nQUERY: '{query}'")
    print("-" * 40)
    
    results = text_collection.query(
        query_embeddings=text_embedder.encode(
            [query],
            normalize_embeddings=True
        ).tolist(),
        n_results=2  # top 2 results per query
    )
    
    for i, doc in enumerate(results['documents'][0]):
        page = results['metadatas'][0][i]['page_num']
        score = 1 - results['distances'][0][i]  # cosine similarity (higher = better)
        print(f"  Rank {i+1} | Page {page} | Similarity {score:.2f}")
        print(f"  Text: {doc[:150].replace(chr(10), ' ')}")
        print()

print("=" * 60)
print("What to look for:")
print("  Similarity > 0.7 = strong match")
print("  Similarity 0.5-0.7 = reasonable match")
print("  Similarity < 0.5 = weak / possibly wrong")


QUERY: 'what fonts are available'
----------------------------------------
  Rank 1 | Page 10 | Similarity 0.75
  Text: 30-pt. -ll pt. 3A  6A. a 784. Alternates Included in fonts, Sizes 14 ta 36 ‘paint.  Alternate Gothic No. 2- No. 77,  Alternate Gothic No. 2 ABCDEFG 12

  Rank 2 | Page 11 | Similarity 0.74
  Text: News Gothic Bold Extended No. 210  News Gothic Bold Extended A  Caps L/e Lic *10-pt.  25A........ 50a *14-pt. ery cee 24a “12-pt.  21AL0 i... 42a  New


QUERY: 'Cooper Black'
----------------------------------------
  Rank 1 | Page 17 | Similarity 0.69
  Text: Cooper Black ABCab 12  Caps L/e Caps Lfe l2-pt. 25A......... 50a 30-pt. 6A.......... lla ld-pt. 21A..... .42a 36-pt. 5A 9a 18-pt. 12A 24a 48-pt. 4A 7a

  Rank 2 | Page 17 | Similarity 0.58
  Text: Clarendon Extend  Caps L/e Caps Lie 14-pt. 12A..........24a 30-pt. 5A... .. 9a 18-pt. 9A... 16a 36-pt. 4A... Ta 24-pt. GA... lla  Cooper Black No. 282


QUERY: 'shipping terms minimum order'
---------------------------------

In [159]:
# cell 8 - BM25 sparse retrieval + RRF fusion with dense results
# BM25 = classic keyword search, scores documents by exact term frequency
# RRF = Reciprocal Rank Fusion - merges two ranked lists into one
# formula: RRF score = sum(1 / (k + rank)) where k=60 is a smoothing constant
# k=60 means even rank-1 results don't dominate too heavily

from rank_bm25 import BM25Okapi

# build BM25 index over all chunks
# tokenize by splitting on whitespace and lowercasing
# simple tokenization works well for this domain
corpus_tokens = [chunk["text"].lower().split() for chunk in all_chunks]
bm25_index = BM25Okapi(corpus_tokens)

print(f"BM25 index built over {len(corpus_tokens)} chunks")

def hybrid_search(query, text_collection, text_embedder, bm25_index, 
                  all_chunks, top_k=20, rrf_k=60):
    
    # --- dense retrieval ---
    query_embedding = text_embedder.encode(
        [query], normalize_embeddings=True
    ).tolist()
    
    dense_results = text_collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    
    # build dense ranking: chunk_id -> rank (1-indexed)
    dense_ranks = {}
    for rank, chunk_id in enumerate(dense_results['ids'][0], start=1):
        dense_ranks[chunk_id] = rank
    
    # --- BM25 sparse retrieval ---
    query_tokens = query.lower().split()
    bm25_scores = bm25_index.get_scores(query_tokens)
    
    # get top_k BM25 results by score
    top_bm25_indices = bm25_scores.argsort()[::-1][:top_k]
    
    # build BM25 ranking: chunk_id -> rank (1-indexed)
    bm25_ranks = {}
    for rank, idx in enumerate(top_bm25_indices, start=1):
        chunk_id = all_chunks[idx]["chunk_id"]
        bm25_ranks[chunk_id] = rank
    
    # --- RRF fusion ---
    # collect all unique chunk_ids from both retrievers
    all_ids = set(dense_ranks.keys()) | set(bm25_ranks.keys())
    
    rrf_scores = {}
    for chunk_id in all_ids:
        score = 0.0
        # add dense contribution (0 if not in dense results)
        if chunk_id in dense_ranks:
            score += 1.0 / (rrf_k + dense_ranks[chunk_id])
        # add BM25 contribution (0 if not in BM25 results)
        if chunk_id in bm25_ranks:
            score += 1.0 / (rrf_k + bm25_ranks[chunk_id])
        rrf_scores[chunk_id] = score
    
    # sort by RRF score descending
    sorted_ids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    # build result list with full chunk data
    results = []
    chunk_lookup = {c["chunk_id"]: c for c in all_chunks}
    
    for chunk_id, rrf_score in sorted_ids[:top_k]:
        if chunk_id in chunk_lookup:
            result = chunk_lookup[chunk_id].copy()
            result["rrf_score"] = rrf_score
            result["dense_rank"] = dense_ranks.get(chunk_id, None)
            result["bm25_rank"] = bm25_ranks.get(chunk_id, None)
            results.append(result)
    
    return results


# test hybrid search on same queries as before
test_queries = [
    "what fonts are available",
    "Cooper Black",
    "shipping terms minimum order",
    "Century Schoolbook sizes",
    "script typefaces",
]

print("\n" + "=" * 60)
for query in test_queries:
    print(f"\nQUERY: '{query}'")
    print("-" * 40)
    
    results = hybrid_search(
        query, text_collection, text_embedder,
        bm25_index, all_chunks, top_k=20
    )
    
    for r in results[:2]:
        print(f"  Page {r['page_num']} | RRF {r['rrf_score']:.4f} | dense_rank={r['dense_rank']} bm25_rank={r['bm25_rank']}")
        print(f"  Text: {r['text'][:120].replace(chr(10), ' ')}")
        print()

print("=" * 60)

BM25 index built over 225 chunks


QUERY: 'what fonts are available'
----------------------------------------
  Page 19 | RRF 0.0308 | dense_rank=5 bm25_rank=5
  Text: Joanna ABCDEFGH]J abcdefghijklmn 1234  Caps Li/e Caps L/e 10-pt. 30A.......... 58a 14-pt. 21A.......... 42a 12-pt. 25A..

  Page 13 | RRF 0.0303 | dense_rank=6 bm25_rank=6
  Text: Caps Caps Le * B-pt. 30A... *10-pt. BSA. .........60a 9-pt. 30A... *12-pt. 20A..........50a  Bell No. 402 (Sold in Fonts


QUERY: 'Cooper Black'
----------------------------------------
  Page 17 | RRF 0.0325 | dense_rank=2 bm25_rank=1
  Text: Clarendon Extend  Caps L/e Caps Lie 14-pt. 12A..........24a 30-pt. 5A... .. 9a 18-pt. 9A... 16a 36-pt. 4A... Ta 24-pt. G

  Page 17 | RRF 0.0323 | dense_rank=1 bm25_rank=3
  Text: Cooper Black ABCab 12  Caps L/e Caps Lfe l2-pt. 25A......... 50a 30-pt. 6A.......... lla ld-pt. 21A..... .42a 36-pt. 5A 


QUERY: 'shipping terms minimum order'
----------------------------------------
  Page 7 | RRF 0.0318 | de

In [160]:
# diagnostic - find which page has Cooper Black and what OCR said
print("Searching for 'Cooper Black' in all chunks:\n")
for chunk in all_chunks:
    if 'cooper' in chunk['text'].lower():
        print(f"Page {chunk['page_num']} | chunk {chunk['chunk_id']}")
        print(chunk['text'][:300])
        print()

Searching for 'Cooper Black' in all chunks:

Page 3 | chunk p3_c13
Chister Text No. 95...
Comstock No. 202... .
Condensed Title Gathic Ne. 43
Condensed Outline No. 123.
Contour No. 40 . .
Cooper Black Na, 282. .
Copperplate Gothic Light No. 340.

Page 17 | chunk p17_c2
Clarendon Extend

Caps L/e Caps Lie
14-pt. 12A..........24a 30-pt. 5A... .. 9a
18-pt. 9A... 16a 36-pt. 4A... Ta
24-pt. GA... lla

Cooper Black No. 282

Cooper Black ABCab 12

Page 17 | chunk p17_c3
Cooper Black ABCab 12

Caps L/e Caps Lfe
l2-pt. 25A......... 50a 30-pt. 6A.......... lla
ld-pt. 21A..... .42a 36-pt. 5A 9a
18-pt. 12A 24a 48-pt. 4A 7a
24.pt. 9A 16a

Duchess No. 844



In [161]:
# cell 9 - cross-encoder reranking
# bi-encoder (bge-small) encodes query and chunk SEPARATELY then compares
# cross-encoder reads query AND chunk TOGETHER - much more accurate but slower
# we use it only on top-20 candidates (not all 236) to keep it fast
# tradeoff: adds ~200-400ms per query but precision improves significantly

from sentence_transformers import CrossEncoder
from src.config import RERANK_TOP_N

print("Loading cross-encoder reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded")

def rerank_results(query, hybrid_results, reranker, top_n=RERANK_TOP_N):
    if not hybrid_results:
        return []
    
    # prepare query-document pairs for cross-encoder
    # cross-encoder scores each (query, document) pair jointly
    pairs = [[query, r["text"]] for r in hybrid_results]
    
    # get relevance scores - higher is more relevant
    scores = reranker.predict(pairs)
    
    # attach scores to results and sort descending
    for i, result in enumerate(hybrid_results):
        result["rerank_score"] = float(scores[i])
    
    reranked = sorted(hybrid_results, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_n]


# test reranking on same queries
test_queries = [
    "what fonts are available",
    "Cooper Black",
    "Century Schoolbook sizes",
    "script typefaces",
]

print("\n" + "=" * 60)
for query in test_queries:
    # step 1: hybrid retrieval (top 20 candidates)
    hybrid_results = hybrid_search(
        query, text_collection, text_embedder,
        bm25_index, all_chunks, top_k=20
    )
    
    # step 2: rerank top 20 down to top 5
    final_results = rerank_results(query, hybrid_results, reranker, top_n=3)
    
    print(f"\nQUERY: '{query}'")
    print("-" * 40)
    for r in final_results:
        print(f"  Page {r['page_num']} | rerank={r['rerank_score']:.3f} | rrf={r['rrf_score']:.4f}")
        print(f"  Text: {r['text'][:120].replace(chr(10), ' ')}")
        print()

print("=" * 60)

Loading cross-encoder reranker...
Reranker loaded


QUERY: 'what fonts are available'
----------------------------------------
  Page 10 | rerank=-1.115 | rrf=0.0164
  Text: 30-pt. -ll pt. 3A  6A. a 784. Alternates Included in fonts, Sizes 14 ta 36 ‘paint.  Alternate Gothic No. 2- No. 77,  Alt

  Page 11 | rerank=-3.448 | rrf=0.0156
  Text: News Gothic Bold ABC 123  JEFFERSON GOTHIC. AABCDEFCHURLMIOPORSY 1234567  36-p 5A 1 Alternates included in fonts, Sizes 

  Page 17 | rerank=-4.209 | rrf=0.0278
  Text: Fourniat No. 403 (Sold in Fonts only)  Fournier ABCDEFG abcdefg 1234  Caps L/c Caps Lfe 14-pt. 21A..........42a 30-pt. 6


QUERY: 'Cooper Black'
----------------------------------------
  Page 17 | rerank=6.527 | rrf=0.0323
  Text: Cooper Black ABCab 12  Caps L/e Caps Lfe l2-pt. 25A......... 50a 30-pt. 6A.......... lla ld-pt. 21A..... .42a 36-pt. 5A 

  Page 3 | rerank=5.693 | rrf=0.0161
  Text: Chister Text No. 95... Comstock No. 202... . Condensed Title Gathic Ne. 43 Condensed Outl

In [162]:
# cell 10 - CLIP image embeddings for visual retrieval
# CLIP encodes page images into 512-dim vectors in the same space as text
# so "show me a bold serif font" (text query) can retrieve specimen page images
# this is the multimodal part - text and images share one embedding space

from sentence_transformers import SentenceTransformer
from src.config import IMAGE_EMBED_MODEL

print("Loading CLIP model...")
# clip-ViT-B-32 wrapped by sentence-transformers for easy use
# first run downloads ~350MB - cached after
clip_model = SentenceTransformer(IMAGE_EMBED_MODEL)
print(f"CLIP model loaded: {IMAGE_EMBED_MODEL}")

# create image collection in Chroma
try:
    chroma_client.delete_collection("page_images")
    print("Deleted existing page_images collection")
except:
    pass

image_collection = chroma_client.create_collection(
    name="page_images",
    metadata={"hnsw:space": "cosine"}
)
print("Created page_images collection")

# embed all page images
print(f"\nEmbedding {len(pages_data)} page images with CLIP...")
print("(first run downloads ~350MB CLIP model - wait for it)")
start = time.time()

# CLIP can encode PIL Images directly
page_images = [p["page_image"] for p in pages_data]
image_embeddings = clip_model.encode(
    page_images,
    batch_size=4,          # small batch - images are large
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"Image embedding done in {time.time() - start:.1f}s")
print(f"Image embedding matrix shape: {image_embeddings.shape}")  # (20, 512)

# store image embeddings in Chroma
image_collection.add(
    ids=[f"page_{p['page_num']}" for p in pages_data],
    embeddings=image_embeddings.tolist(),
    metadatas=[{
        "page_num": p["page_num"],
        "quality_score": p["quality_score"],
        "text_source": p["text_source"],
    } for p in pages_data]
)

print(f"\nStored {image_collection.count()} page images in Chroma")

# test visual retrieval with text queries
print("\nTesting visual retrieval:")
print("=" * 50)

visual_queries = [
    "bold black typeface",
    "script cursive handwriting font",
    "geometric sans serif",
    "decorative ornamental arrows",
]

for query in visual_queries:
    # encode text query with CLIP text encoder
    query_embedding = clip_model.encode(
        [query],
        normalize_embeddings=True
    ).tolist()
    
    results = image_collection.query(
        query_embeddings=query_embedding,
        n_results=2
    )
    
    print(f"\nQUERY: '{query}'")
    for i, meta in enumerate(results['metadatas'][0]):
        score = 1 - results['distances'][0][i]
        print(f"  Page {meta['page_num']} | similarity {score:.3f}")

Loading CLIP model...


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


CLIP model loaded: clip-ViT-B-32
Deleted existing page_images collection
Created page_images collection

Embedding 19 page images with CLIP...
(first run downloads ~350MB CLIP model - wait for it)


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Image embedding done in 1.0s
Image embedding matrix shape: (19, 512)

Stored 19 page images in Chroma

Testing visual retrieval:

QUERY: 'bold black typeface'
  Page 17 | similarity 0.275
  Page 6 | similarity 0.269

QUERY: 'script cursive handwriting font'
  Page 18 | similarity 0.230
  Page 14 | similarity 0.226

QUERY: 'geometric sans serif'
  Page 10 | similarity 0.258
  Page 9 | similarity 0.256

QUERY: 'decorative ornamental arrows'
  Page 7 | similarity 0.240
  Page 6 | similarity 0.238


In [163]:
# cell 11 - query router + answer generation
# router reads the query and decides: text retrieval, image retrieval, or both
# then generates an answer from retrieved context using Groq LLM

from groq import Groq
from src.config import GROQ_API_KEY, GROQ_MODEL

groq_client = Groq(api_key=GROQ_API_KEY)

def route_query(query, groq_client, model=GROQ_MODEL):
    # ask the LLM to classify the query intent
    # we give it three choices and ask for one word answer
    # simple classification task - small model handles this fine
    
    prompt = f"""Classify this query into exactly one category:
- TEXT: asking for facts, names, prices, specifications, descriptions
- IMAGE: asking to see, show, display, or visually find something  
- BOTH: needs both factual information AND visual display

Query: "{query}"

Reply with only one word: TEXT, IMAGE, or BOTH"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0,  # deterministic - routing must be consistent
    )
    
    route = response.choices[0].message.content.strip().upper()
    
    # safety net - if LLM returns something unexpected, default to BOTH
    if route not in ["TEXT", "IMAGE", "BOTH"]:
        route = "BOTH"
    
    return route


def generate_answer(query, text_chunks, groq_client, model=GROQ_MODEL):
    if not text_chunks:
        return "No relevant text found for this query."
    
    context_parts = []
    for i, chunk in enumerate(text_chunks):
        context_parts.append(
            f"[Source: Page {chunk['page_num']}]\n{chunk['text']}"
        )
    context = "\n\n".join(context_parts)
    
    prompt = f"""You are an expert on the Los Angeles Type Founders catalog.
The context below is OCR-extracted from a scanned document so it contains noise:
- 'l2-pt' means 12-pt, 'ld-pt' means 14-pt, 'lla' means 11a
- dots like '.......' are just spacing artifacts, ignore them
- L/e and L/c both mean lowercase count

Answer the question using the provided context despite the OCR noise.
Always cite the page number. If genuinely not found, say so.

Context:
{context}

Question: {query}

Answer:"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0.1,
    )
    
    return response.choices[0].message.content.strip()

def full_pipeline(query):
    print(f"Query: '{query}'")
    
    # step 1: route
    route = route_query(query, groq_client)
    print(f"Route: {route}")
    
    text_results = []
    image_results = []
    
    # step 2: retrieve based on route
    if route in ["TEXT", "BOTH"]:
        hybrid_results = hybrid_search(
            query, text_collection, text_embedder,
            bm25_index, all_chunks, top_k=20
        )
        text_results = rerank_results(query, hybrid_results, reranker, top_n=5)
        print(f"Text results: {len(text_results)} chunks retrieved")
    
    if route in ["IMAGE", "BOTH"]:
        query_embedding = clip_model.encode(
            [query], normalize_embeddings=True
        ).tolist()
        img_results = image_collection.query(
            query_embeddings=query_embedding,
            n_results=3
        )
        image_results = [
            {
                "page_num": meta["page_num"],
                "similarity": 1 - dist,
                "page_image": pages_data[
                    next(i for i, p in enumerate(pages_data) 
                         if p["page_num"] == meta["page_num"])
                ]["page_image"]
            }
            for meta, dist in zip(
                img_results['metadatas'][0],
                img_results['distances'][0]
            )
        ]
        print(f"Image results: {len(image_results)} pages retrieved")
    
    # step 3: generate answer from text context
    if text_results:
        answer = generate_answer(query, text_results, groq_client)
    else:
        answer = "Visual query — see retrieved images below."
    
    print(f"\nAnswer: {answer}")
    
    if image_results:
        print(f"\nTop image match: Page {image_results[0]['page_num']} "
              f"(similarity: {image_results[0]['similarity']:.3f})")
    
    return {
        "query": query,
        "route": route,
        "answer": answer,
        "text_chunks": text_results,
        "image_results": image_results,
    }


# test the full pipeline
test_queries = [
    "What point sizes is Cooper Black available in?",
    "Show me a bold typeface specimen",
    "What is the minimum order value?",
]

print("=" * 60)
for query in test_queries:
    result = full_pipeline(query)
    print("=" * 60)

Query: 'What point sizes is Cooper Black available in?'


NotFoundError: Error code: 404 - {'error': {'message': 'The model `qwen/qwen3.6-27b` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [ ]:
# debug - call groq directly with Cooper Black context
test_context = """[Source: Page 17]
Cooper Black ABCab 12

Caps L/e Caps Lfe
l2-pt. 25A......... 50a 30-pt. 6A.......... lla
ld-pt. 21A..... .42a 36-pt. 5A 9a
18-pt. 12A 24a 48-pt. 4A 7a
24.pt. 9A 16a"""

test_prompt = f"""You are an expert on the Los Angeles Type Founders catalog.
OCR noise guide: l2-pt=12pt, ld-pt=14pt, 24.pt=24pt, lla=11a
List the point sizes available for Cooper Black from this context.

Context:
{test_context}

Question: What point sizes is Cooper Black available in?

Answer:"""

response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": test_prompt}],
    max_tokens=300,
    temperature=0.1,
)

print(f"Raw response: '{response.choices[0].message.content}'")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Model used: {response.model}")

Raw response: ''
Finish reason: length
Model used: openai/gpt-oss-20b


In [ ]:
GROQ_MODEL = "qwen/qwen3.8-27b"

groq_client = Groq(api_key=GROQ_API_KEY)

# verify Cooper Black answer now works
response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": test_prompt}],
    max_tokens=300,
    temperature=0.1,
)
print(f"Response: '{response.choices[0].message.content}'")
print(f"Finish reason: {response.choices[0].finish_reason}")

Response: 'Based on the provided context from the Los Angeles Type Founders catalog (Page 17), the point sizes listed for Cooper Black are:

*   **12-pt** (OCR: l2-pt)
*   **14-pt** (OCR: ld-pt)
*   **18-pt**
*   **24-pt** (OCR: 24.pt)
*   **30-pt**
*   **36-pt**
*   **48-pt**

Therefore, Cooper Black is available in **12, 14, 18, 24, 30, 36, and 48 point sizes**.'
Finish reason: stop


In [164]:
# cell 11 - full pipeline with fixed model
GROQ_MODEL = "qwen/qwen3.8-27b"
groq_client = Groq(api_key=GROQ_API_KEY)

def route_query(query, groq_client, model=GROQ_MODEL):
    prompt = f"""Classify this query into exactly one category:
- TEXT: asking for facts, names, prices, specifications, descriptions
- IMAGE: asking to see, show, display, or visually find something  
- BOTH: needs both factual information AND visual display

Query: "{query}"

Reply with only one word: TEXT, IMAGE, or BOTH"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0,
    )
    route = response.choices[0].message.content.strip().upper()
    if route not in ["TEXT", "IMAGE", "BOTH"]:
        route = "BOTH"
    return route


def generate_answer(query, text_chunks, groq_client, model=GROQ_MODEL):
    if not text_chunks:
        return "No relevant text found for this query."
    
    context_parts = []
    for i, chunk in enumerate(text_chunks):
        context_parts.append(
            f"[Source: Page {chunk['page_num']}]\n{chunk['text']}"
        )
    context = "\n\n".join(context_parts)
    
    prompt = f"""You are an expert on the Los Angeles Type Founders catalog.
OCR noise guide: l2-pt=12pt, ld-pt=14pt, 24.pt=24pt, lla=11a, dots are spacing artifacts.
Answer using ONLY the provided context. Cite page numbers.
If not found say: Not found in indexed pages.

Context:
{context}

Question: {query}

Answer:"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0.1,
    )
    return response.choices[0].message.content.strip()


def full_pipeline(query):
    print(f"Query: '{query}'")
    route = route_query(query, groq_client)
    print(f"Route: {route}")
    
    text_results = []
    image_results = []
    
    if route in ["TEXT", "BOTH"]:
        hybrid_results = hybrid_search(
            query, text_collection, text_embedder,
            bm25_index, all_chunks, top_k=20
        )
        text_results = rerank_results(query, hybrid_results, reranker, top_n=5)
        print(f"Text results: {len(text_results)} chunks")
    
    if route in ["IMAGE", "BOTH"]:
        query_embedding = clip_model.encode(
            [query], normalize_embeddings=True
        ).tolist()
        img_results = image_collection.query(
            query_embeddings=query_embedding,
            n_results=3
        )
        image_results = [
            {
                "page_num": meta["page_num"],
                "similarity": 1 - dist,
                "page_image": pages_data[
                    next(i for i, p in enumerate(pages_data)
                         if p["page_num"] == meta["page_num"])
                ]["page_image"]
            }
            for meta, dist in zip(
                img_results['metadatas'][0],
                img_results['distances'][0]
            )
        ]
        print(f"Image results: {len(image_results)} pages")
    
    if text_results:
        answer = generate_answer(query, text_results, groq_client)
    else:
        answer = "Visual query — see retrieved images below."
    
    print(f"\nAnswer: {answer}")
    if image_results:
        print(f"Top image: Page {image_results[0]['page_num']} "
              f"(similarity: {image_results[0]['similarity']:.3f})")
    
    return {
        "query": query,
        "route": route,
        "answer": answer,
        "text_chunks": text_results,
        "image_results": image_results,
    }


# test
test_queries = [
    "What point sizes is Cooper Black available in?",
    "Show me a bold typeface specimen",
    "What fonts start with letter C?",
]

print("=" * 60)
for query in test_queries:
    result = full_pipeline(query)
    print("=" * 60)

Query: 'What point sizes is Cooper Black available in?'
Route: TEXT
Text results: 5 chunks

Answer: Based on the provided context, Cooper Black is available in the following point sizes:

*   **12-pt** (l2-pt)
*   **14-pt** (ld-pt)
*   **18-pt**
*   **24-pt** (24.pt)
*   **30-pt**
*   **36-pt**
*   **48-pt**

These sizes are listed in the specifications for "Cooper Black ABCab 12" on **Page 17**.
Query: 'Show me a bold typeface specimen'
Route: IMAGE
Image results: 3 pages

Answer: Visual query — see retrieved images below.
Top image: Page 6 (similarity: 0.309)
Query: 'What fonts start with letter C?'
Route: TEXT
Text results: 5 chunks

Answer: Not found in indexed pages.
